In [ ]:
import duckdb
import logging
from pathlib import Path
from typing import Optional

# Configure standard logging (Industry Best Practice)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def extract_behavioral_features(user_input_path: str, dev_mode: bool = False) -> str:
    """
    Securely extracts a time-bucketed behavioral feature matrix from raw logon logs.
    
    Args:
        user_input_path (str): Filename (in PROD) or full local path (in DEV).
        dev_mode (bool): If True, bypasses strict sandbox for local testing.
        
    Returns:
        str: The absolute file path to the generated Parquet feature matrix.
        
    Raises:
        ValueError: If a security violation (SQLi/Traversal) is detected.
        FileNotFoundError: If the target log file does not exist.
    """
    
    logging.info("Initiating Feature Extraction Pipeline...")

    # =====================================================================
    # 1. THE SECURITY SANDBOX & PATH VALIDATION
    # =====================================================================
    if dev_mode:
        logging.warning("Running in DEV_MODE. Strict security sandbox is DISABLED.")
        target_file: Path = Path(user_input_path).resolve()
        
        if not target_file.exists():
            logging.error(f"DEV ERROR: Target file not found at {target_file}")
            raise FileNotFoundError(f"Cannot find file at {target_file}")
            
    else:
        # PROD MODE: Enforce the Sandbox
        safe_directory: Path = Path("/path/to/your/secure/datasets/").resolve()
        target_file: Path = (safe_directory / user_input_path).resolve()
        
        if not target_file.is_relative_to(safe_directory):
            logging.error(f"SECURITY ALERT: Directory Traversal Attempt Detected: {user_input_path}")
            raise ValueError("Access Denied: Invalid file path.")
            
    if target_file.suffix != '.csv':
        logging.error(f"SECURITY ALERT: Invalid file type attempted: {target_file.suffix}")
        raise ValueError("Access Denied: Only .csv files are permitted.")

    # Define the secure output path for the Parquet file
    output_dir: Path = Path("/path/to/your/secure/features/").resolve()
    # Create the directory if it doesn't exist (useful for first-time runs)
    output_dir.mkdir(parents=True, exist_ok=True) 
    
    output_file: Path = output_dir / "master_behavioral_features.parquet"

    # =====================================================================
    # 2. THE SQL ARCHITECTURE (5 Core Features)
    # =====================================================================
    logging.info(f"Ingesting raw logs from: {target_file}")
    logging.info("Applying Time-Bucketed CTE and extracting 5 behavioral features...")
    
    query: str = f"""
    COPY (
        -- STEP 1: Deduplication & Sessionization
        WITH SessionizedLogs AS (
            SELECT 
                user,
                pc,      -- Grab the PC ID for Lateral Movement detection
                activity, -- Grab the activity for Failed Login detection
                
                -- Snap timestamps to a 5-minute grid & fix timezones
                TIME_BUCKET(
                    INTERVAL '5 minutes', 
                    TRY_CAST(date AS TIMESTAMP)
                ) AS session_time
                
            FROM read_csv_auto('{target_file}')
            WHERE user != 'SYSTEM'
                AND (activity = 'Logon' OR activity ILIKE '%fail%')
              -- We do NOT filter for only 'Logon' here anymore, because we need to see the 'Failed' attempts too!
            GROUP BY user, pc, activity, session_time
        )
        
        -- STEP 2: Feature Engineering Aggregation
        SELECT 
            user, 
            
            -- Feature 1: Total Volume (Authentication Velocity)
            COUNT(*) AS total_events,
            
            -- Feature 2 & 3: Off-Hours Ratio
            SUM(
                CASE 
                    WHEN DAYOFWEEK(session_time) IN (0, 6) THEN 1
                    WHEN EXTRACT(hour FROM session_time) < 6 OR EXTRACT(hour FROM session_time) >= 18 THEN 1 
                    ELSE 0 
                END
            ) AS off_hours_events,
            
            SUM(
                CASE 
                    WHEN DAYOFWEEK(session_time) IN (0, 6) THEN 1
                    WHEN EXTRACT(hour FROM session_time) < 6 OR EXTRACT(hour FROM session_time) >= 18 THEN 1 
                    ELSE 0 
                END
            ) * 1.0 / NULLIF(COUNT(*), 0) AS off_hours_ratio,
            
            -- Feature 4: Lateral Movement (Unique Systems)
            COUNT(DISTINCT pc) AS unique_systems_accessed,
            
            -- Feature 5: Brute Force (Failed Login Ratio)
            SUM(CASE WHEN activity LIKE '%Fail%' THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS failed_login_ratio

        FROM SessionizedLogs
        GROUP BY user
        
    ) TO '{output_file}' (FORMAT PARQUET);
    """
    
    # =====================================================================
    # 3. EXECUTION
    # =====================================================================
    try:
        duckdb.sql(query)
        logging.info(f"Pipeline Complete. Feature matrix securely saved to: {output_file}")
        return str(output_file)
    except Exception as e:
        logging.error(f"Database Execution Failed: {str(e)}")
        raise e

In [ ]:
result = extract_behavioral_features('CERT_dataset/r4.2/logon.csv', dev_mode=True)